In [ ]:
# ===== Colab环境配置 =====
# 运行此cell安装所有依赖（约2-3分钟）
!pip install -q numpy scipy matplotlib scikit-learn tqdm
!pip install -q torch torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q librosa soundfile tensorboard
!apt-get install -qq ffmpeg sox
print('环境配置完成！')

# 确认GPU
import torch
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('GPU不可用，请在运行时类型中选择T4 GPU')

# 下载ESC-50数据集（如果尚未下载）
import os
if not os.path.exists('ESC-50'):
    print('正在下载ESC-50数据集（约60MB）...')
    !wget -q https://github.com/karoldvl/ESC-50/archive/master.zip -O esc50.zip
    !unzip -q esc50.zip
    !mv ESC-50-master ESC-50
    !rm esc50.zip
    print('ESC-50下载完成！')
else:
    print('ESC-50已存在')


# CRNN音频分类器

## 学习目标

- 用PyTorch搭建完整的音频分类pipeline
- 理解CRNN架构：CNN提取时频特征 + RNN建模时序 + FC分类
- 理解CNN -> CRNN -> Attention的架构演进
- 掌握评估指标：混淆矩阵、精确率、召回率、F1

## 1. PyTorch Dataset/DataLoader for Audio

音频数据加载的关键思路：
1. Dataset负责逐条加载音频 -> 提取特征 -> 返回(特征, 标签)
2. DataLoader负责批量组合多个样本
3. 特征提取在__getitem__中完成

```
WAV文件 -> Dataset.__getitem__() -> (Mel Spectrogram [n_mels, T], label)
                                          | DataLoader
                               (batch [B, n_mels, T], batch_labels [B])
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torchaudio
import torchaudio.transforms as T
import os
import pandas as pd

plt.rcParams['font.size'] = 12

ESC50_DIR = 'ESC-50'
ESC50_AVAILABLE = os.path.exists(os.path.join(ESC50_DIR, 'audio'))
if ESC50_AVAILABLE:
    df = pd.read_csv(os.path.join(ESC50_DIR, 'meta', 'esc50.csv'))
    print('ESC-50已加载, 共%d条, %d类' % (len(df), df['category'].nunique()))
else:
    print('ESC-50未找到, 将使用合成数据')

In [ ]:
class ESC50Dataset(torch.utils.data.Dataset):
    """ESC-50数据集
    输出: features [n_mels, max_frames], label int, length int
    """
    def __init__(self, meta_csv, audio_dir, sr=16000, n_mels=64, n_fft=512,
                 hop_length=160, fold=None, subset='train'):
        self.df = pd.read_csv(meta_csv)
        self.audio_dir = audio_dir; self.sr = sr
        self.mel_transform = T.MelSpectrogram(sr, n_fft=n_fft, hop_length=hop_length, n_mels=n_mels)
        if fold is not None:
            if subset == 'train': self.df = self.df[self.df['fold'] != fold].reset_index(drop=True)
            else: self.df = self.df[self.df['fold'] == fold].reset_index(drop=True)
        self.classes = sorted(self.df['category'].unique().tolist())
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        wav, orig_sr = torchaudio.load(os.path.join(self.audio_dir, row['filename']))
        if orig_sr != self.sr: wav = T.Resample(orig_sr, self.sr)(wav)
        if wav.shape[0] > 1: wav = wav.mean(dim=0, keepdim=True)
        mel = self.mel_transform(wav)
        mel_db = torchaudio.functional.amplitude_to_DB(mel, top_db=80)
        mel_db = (mel_db - mel_db.mean()) / (mel_db.std() + 1e-8)
        label = self.class_to_idx[row['category']]
        length = mel_db.shape[-1]
        return mel_db.squeeze(0), label, length

if ESC50_AVAILABLE:
    dataset = ESC50Dataset(os.path.join(ESC50_DIR,'meta','esc50.csv'), os.path.join(ESC50_DIR,'audio'), fold=1, subset='train')
    feat, label, length = dataset[0]
    print('特征形状:', feat.shape, '标签:', label, '有效帧数:', length)

In [ ]:
# 处理变长序列: Padding
def collate_fn(batch):
    feats, labels, lengths = zip(*batch)
    max_len = max(lengths); n_mels = feats[0].shape[0]
    padded = torch.zeros(len(batch), n_mels, max_len)
    for i, (feat, length) in enumerate(zip(feats, lengths)):
        padded[i, :, :length] = feat[:, :length]
    return padded, torch.LongTensor(labels), torch.LongTensor(lengths)

if ESC50_AVAILABLE:
    loader = torch.utils.data.DataLoader(dataset, batch_size=16, shuffle=True, collate_fn=collate_fn)
    bf, bl, blen = next(iter(loader))
    print('Batch特征:', bf.shape, '标签:', bl.shape)

## 2. 模型架构选择

| 架构 | 时间维度处理 | 特点 |
|------|-------------|------|
| CNN | 忽略时序 | 简单, 适合短音频 |
| CRNN | RNN建模时序 | 兼顾局部+全局, **最实用** |
| Attention | 聚焦关键帧 | 更灵活但更复杂 |

CRNN数据流:
```
语谱图 [n_mels, T] -> CNN(局部模式) -> [hidden, T'] -> GRU(时序) -> FC(分类) -> [n_classes]
```

In [ ]:
class CRNN(nn.Module):
    def __init__(self, n_mels=64, n_classes=50, cnn_channels=[16,32,64],
                 gru_hidden=64, gru_layers=2, dropout=0.3):
        super().__init__()
        cnn_layers = []; in_ch = 1
        for out_ch in cnn_channels:
            cnn_layers.extend([
                nn.Conv2d(in_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch),
                nn.ReLU(), nn.MaxPool2d((2, 1)),  # 只在频率维度下采样
            ])
            in_ch = out_ch
        self.cnn = nn.Sequential(*cnn_layers)
        cnn_out_freq = n_mels
        for _ in cnn_channels: cnn_out_freq = cnn_out_freq // 2
        self.gru = nn.GRU(cnn_channels[-1]*cnn_out_freq, gru_hidden, gru_layers,
                          batch_first=True, bidirectional=True,
                          dropout=dropout if gru_layers > 1 else 0)
        self.classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(gru_hidden*2, n_classes))
    def forward(self, x, lengths=None):
        x = x.unsqueeze(1)  # [B,1,n_mels,T]
        cnn_out = self.cnn(x)  # [B,C,F,T]
        b,c,f,t = cnn_out.shape
        cnn_out = cnn_out.view(b,c*f,t).permute(0,2,1)  # [B,T,C*F]
        gru_out, _ = self.gru(cnn_out)
        if lengths is not None:
            idx = (lengths-1).view(-1,1,1).expand(-1,1,gru_out.size(2))
            last = gru_out.gather(1, idx.to(gru_out.device)).squeeze(1)
        else:
            last = gru_out[:, -1, :]
        return self.classifier(last)

model = CRNN(n_mels=64, n_classes=10)
x = torch.randn(4, 64, 100)
print('输入:', x.shape, '输出:', model(x).shape)
print('参数总数: %d' % sum(p.numel() for p in model.parameters()))

### 为什么需要RNN?

CNN只看局部模式, 不知道模式的先后顺序。但声音是时序信号——事件先后顺序很重要。
CRNN = CNN找"模式" + RNN理解"模式的顺序"。

## 3. 训练与评估

In [ ]:
if ESC50_AVAILABLE:
    selected = ['dog','rooster','crying_baby','chainsaw','crackling_fire',
                'helicopter','rain','sea_waves','siren','vacuum_cleaner']
    df10 = df[df['category'].isin(selected)].reset_index(drop=True)
    df10.to_csv('/tmp/esc10.csv', index=False)
    train_ds = ESC50Dataset('/tmp/esc10.csv', os.path.join(ESC50_DIR,'audio'), fold=1, subset='train')
    val_ds = ESC50Dataset('/tmp/esc10.csv', os.path.join(ESC50_DIR,'audio'), fold=1, subset='val')
    n_classes = len(selected)
    train_loader = torch.utils.data.DataLoader(train_ds, batch_size=16, shuffle=True, collate_fn=collate_fn)
    val_loader = torch.utils.data.DataLoader(val_ds, batch_size=16, collate_fn=collate_fn)
    model = CRNN(n_mels=64, n_classes=n_classes)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    train_losses, val_accs = [], []
    for epoch in range(20):
        model.train(); total_loss = 0
        for feats, labels, lengths in train_loader:
            out = model(feats, lengths); loss = criterion(out, labels)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            total_loss += loss.item()
        train_losses.append(total_loss / len(train_loader))
        model.eval(); correct = total = 0
        with torch.no_grad():
            for feats, labels, lengths in val_loader:
                correct += (model(feats, lengths).argmax(1) == labels).sum().item()
                total += labels.size(0)
        val_accs.append(correct / total)
        if (epoch+1) % 5 == 0: print('Epoch %2d, Loss: %.4f, Val Acc: %.4f' % (epoch+1, train_losses[-1], val_accs[-1]))
    print('最终验证准确率: %.4f' % val_accs[-1])
else:
    print('ESC-50不可用')

### 评估指标

| 指标 | 含义 |
|------|------|
| 混淆矩阵 | 每个类别被分对/分错的详细情况 |
| 精确率(Precision) | 预测为某类中真正是该类的比例 |
| 召回率(Recall) | 真正是某类中被正确预测的比例 |
| F1 | 精确率和召回率的调和平均 |

In [ ]:
if ESC50_AVAILABLE:
    from sklearn.metrics import confusion_matrix, classification_report
    model.eval(); all_preds, all_labels = [], []
    with torch.no_grad():
        for feats, labels, lengths in val_loader:
            all_preds.extend(model(feats, lengths).argmax(1).tolist())
            all_labels.extend(labels.tolist())
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(10, 8))
    plt.imshow(cm, cmap='Blues'); plt.colorbar()
    plt.xticks(range(n_classes), selected, rotation=45, ha='right')
    plt.yticks(range(n_classes), selected)
    plt.xlabel('Predicted'); plt.ylabel('True'); plt.title('Confusion Matrix')
    plt.tight_layout(); plt.show()
    print(classification_report(all_labels, all_preds, target_names=selected))
else:
    print('ESC-50不可用')

## 4. 总结

| 方面 | 图像分类 | 音频分类 |
|------|----------|----------|
| 输入 | 像素矩阵 | 语谱图 |
| 变长问题 | 通常固定尺寸 | 需padding或截断 |
| 时序性 | 空间关系 | 时间关系(RNN有帮助) |
| 数据增强 | 翻转、裁剪 | 加噪、频谱掩蔽 |

## 本节要点

1. Dataset/DataLoader是PyTorch数据加载的标准模式——音频需处理变长序列
2. CRNN = CNN + RNN + FC：CNN提取局部模式，RNN建模时序，FC分类
3. MaxPool只在频率维度下采样，保留时间维度给RNN
4. 评估不只看准确率：混淆矩阵、精确率、召回率、F1
5. CRNN是音频分类的workhorse，也是理解DeepACE/DeepFilterNet的基础

---
下一节：[03-ci-tasks.ipynb](03-ci-tasks.ipynb)